
Este notebook documenta el flujo de un orquestador de consultas que clasifica por dominio y enruta cada pregunta a un agente RAG especializado. Usa el código existente en `src/` y cumple los requisitos de la consigna y los entregables.
Secciones:
1. Setup e imports
2. Carga de documentos y vector stores
3. Definición de agentes y orquestador
4. Enrutamiento y pruebas
5. Instrumentación con Langfuse

In [ ]:
# Setup e imports
import json
import os
import sys
from pathlib import Path

repo_root = Path(os.getcwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.config import Settings
from src.data_loader import build_domain_retrievers
from src.orchestrator import Orchestrator
from src.evaluator import ResponseEvaluator

print(f"Repositorio: {repo_root}")
print("Imports completos")

## Carga de documentos y vector stores

Esta sección prepara los datos de conocimiento por dominio y construye los retrievers de FAISS. El proyecto está configurado para usar documentos de ejemplo en `data/` y realiza chunking con `RecursiveCharacterTextSplitter`.


In [ ]:
# Asegura que existan los archivos de datos de ejemplo
from pathlib import Path

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

example_files = {
    "hr_docs.txt": "Política de vacaciones: los empleados tienen derecho a 20 días hábiles de licencia al año.\nPreguntas sobre nómina deben remitirse al área de Finanzas.\n",
    "it_docs.txt": "Procedimiento de soporte: reiniciar el equipo y verificar la conexión de red.\nEl equipo IT no gestiona solicitudes de nómina.\n",
    "finance_docs.txt": "La política de gastos permite reembolso de viáticos con factura válida.\nEl presupuesto anual se revisa en el primer trimestre.\n",
    "legal_docs.txt": "El área legal revisa contratos y cumplimiento normativo.\nLas consultas de reclamos laborales deben derivarse a RRHH cuando hay empleados involucrados.\n",
}

for file_name, content in example_files.items():
    path = data_dir / file_name
    if not path.exists():
        path.write_text(content, encoding="utf-8")
        print(f"Creado: {path}")

settings = Settings()
print("Configuración leída de .env / entorno")

try:
    retrievers = build_domain_retrievers(settings)
    print("Retrievers construidos:", list(retrievers.keys()))
except Exception as exc:
    print("Error al construir retrievers:", exc)

## Definición del orquestador y agentes RAG

Aquí se construye el `Orchestrator` que clasifica la consulta en dominios (`HR`, `IT Support`, `Finance`, `Legal`) y enruta cada pregunta al agente RAG especializado. El evaluador de calidad también se instancia en esta sección.

In [ ]:
# Construcción del orquestador y del evaluador
orchestrator = Orchestrator(settings, retrievers)
evaluator = ResponseEvaluator(settings)

print("Orquestador listo")
print("Dominios disponibles:", list(orchestrator.qa_chains.keys()))

## Enrutamiento de consultas y pruebas

Se prueban varias consultas de ejemplo que cubren todos los dominios esperados. Esto demuestra cómo el orquestador clasifica y enruta usando los agentes especializados, y cómo el evaluador puntúa cada respuesta.


In [ ]:
# Test queries con dominio esperado
queries = [
    {"query": "¿Cuántos días de vacaciones me corresponden este año?", "expected_domain": "HR"},
    {"query": "Mi laptop no arranca y muestra error de disco, ¿qué debo hacer?", "expected_domain": "IT Support"},
    {"query": "¿Puedo reclamar viáticos sin factura?", "expected_domain": "Finance"},
    {"query": "¿Cómo gestionamos reclamaciones de empleados con el área legal?", "expected_domain": "Legal"},
    {"query": "Necesito cambiar mi cuenta bancaria para el pago de sueldo.", "expected_domain": "HR"},
    {"query": "La red Wi-Fi de la oficina no conecta en el tercer piso.", "expected_domain": "IT Support"},
    {"query": "¿Cuál es el proceso para presentar un presupuesto anual?", "expected_domain": "Finance"},
    {"query": "¿Qué documentación exige legal para revisar contratos de proveedor?", "expected_domain": "Legal"},
    {"query": "Tengo dudas sobre mi liquidación y finiquitos.", "expected_domain": "HR"},
    {"query": "¿Podemos usar herramientas de terceros para backups?", "expected_domain": "IT Support"},
]

results = []
for item in queries:
    query = item["query"]
    result = orchestrator.route_query(query)
    evaluation = evaluator.evaluate(query, result["answer"])
    results.append({
        "query": query,
        "expected_domain": item["expected_domain"],
        "predicted_domain": result["domain"],
        "answer": result["answer"],
        "sources": result["sources"],
        "evaluation": evaluation,
    })

for row in results:
    print("---")
    print("Consulta:", row["query"])
    print("Esperado:", row["expected_domain"], "| Predicho:", row["predicted_domain"])
    print("Respuesta:", row["answer"])
    print("Fuentes:", row["sources"])
    print("Evaluación:", row["evaluation"])

## Integración con Langfuse

El proyecto usa `LangfuseTracer` para capturar el flujo de ejecución de LangChain. Si `LANGFUSE_API_KEY` y `LANGFUSE_BASE_URL` están definidos, todas las llamadas a los agentes y el evaluador quedarán trazadas en Langfuse.

En esta sección se muestra cómo verificar que los callbacks se han conectado correctamente.

In [ ]:
# Verificar la integración con Langfuse en el callback manager
callback_manager = settings.create_callback_manager()
print("Callback manager creado")
print("Tipo:", type(callback_manager))

- El notebook reutiliza la estructura de `src/` para separar configuración, carga de datos, orquestación y evaluación.
- El orquestador clasifica primero y luego utiliza `RetrievalQA` para garantizar respuestas fundadas en la documentación interna.
- El evaluador produce una puntuación de calidad de 1 a 10, que puede mapearse a métricas de calidad en Langfuse.
- Para cumplir la consigna, el sistema ofrece trazabilidad, enrutamiento condicional y un agente evaluador bonus.
